# 07 Barcode Measurement

**Purpose:** Compute quantitative barcode measurements only for scans or B-scans classified as barcode-positive.

**Inputs:**
- `data/processed/roi/`: preprocessed below-RPE ROI volumes.
- `data/processed/predictions/barcode_predictions.csv`: ResNet barcode predictions from notebook 06.
- `data/processed/labels/clinician/barcode_labels.csv`: clinician labels if available.
- Optional Grad-CAM maps or predicted positive B-scan ranges from notebook 06.

**Main task:**
For barcode-positive scans, estimate barcode extent and summarize it into interpretable quantitative variables.

**Planned workflow:**
1. Load ROI volumes and positive predictions/labels.
2. Select positive volumes/B-scans.
3. Use model localization, Grad-CAM, thresholding, or post-processing to identify barcode-positive regions.
4. Compute measurements:
   - area fraction,
   - total barcode width,
   - number of bars,
   - mean bar width,
   - median bar width,
   - maximum bar width,
   - bar density,
   - positive B-scan count,
   - positive B-scan fraction.
5. Summarize measurements at both B-scan and volume level.
6. Save measurement tables for downstream validation and clinical analysis.

**Code organization:**
- Long measurement functions go in `src/barcode/measurement.py`.
- Any mask/post-processing functions used specifically for measurement go in `src/barcode/measurement.py` unless they become large enough to split later.
- The notebook should only call high-level measurement functions and inspect output tables.

**Expected outputs:**
- `data/processed/features/barcode_bscan_measurements.csv`
- `data/processed/features/barcode_volume_measurements.csv`
- `data/processed/figures/measurement_examples/`

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PREDICTIONS_FILE = PROCESSED_DIR / "predictions" / "barcode_volume_predictions.csv"
LABEL_FILE = PROCESSED_DIR / "labels" / "clinician" / "barcode_labels.csv"

In [ ]:
from barcode.measurement import run_measurement_pipeline

In [ ]:
if PREDICTIONS_FILE.exists():
    print("Using ResNet predictions:", PREDICTIONS_FILE)
    source_file = PREDICTIONS_FILE
    source_type = "predictions"
elif LABEL_FILE.exists():
    print("Predictions not found. Using clinician labels:", LABEL_FILE)
    source_file = LABEL_FILE
    source_type = "labels"
else:
    print("No predictions or labels found yet.")
    source_file = None
    source_type = None

In [ ]:
if source_type == "predictions":
    bscan_measure_df, volume_measure_df = run_measurement_pipeline(
        processed_dir=PROCESSED_DIR,
        predictions_file=source_file,
        max_volumes=5,
    )
elif source_type == "labels":
    bscan_measure_df, volume_measure_df = run_measurement_pipeline(
        processed_dir=PROCESSED_DIR,
        label_file=source_file,
        max_volumes=5,
    )

In [ ]:
if source_file is not None:
    display(bscan_measure_df.head())
    display(volume_measure_df.head())

In [ ]:
if source_file is not None:
    print("B-scan measurement rows:", len(bscan_measure_df))
    print("Volume measurement rows:", len(volume_measure_df))

In [ ]:
for path in [
    PROCESSED_DIR / "features" / "barcode_bscan_measurements.csv",
    PROCESSED_DIR / "features" / "barcode_volume_measurements.csv",
]:
    print(path, "exists:", path.exists())